In [ ]:
import torch
import matplotlib.pyplot as plt

from core import RNNConfig, MotorCortexRNN

# ----------------------------
# Build model
# ----------------------------
cfg = RNNConfig(
    n_inp=10,
    n_rec=200,
    dt=0.01,
    tau=0.1,
    noise_std=0.01,
    device="cpu",
)

model = MotorCortexRNN(cfg)

# ----------------------------
# Make one input trial
# ----------------------------
T = 100
x = torch.zeros(1, T, cfg.n_inp)  # [batch=1, time, input_dim]

# Example piecewise-constant input:
# first 30 steps baseline, next 40 steps stronger drive, last 30 steps back down
x[:, :30, 0] = 0.2
x[:, 30:70, 0] = 1.0
x[:, 70:, 0] = 0.4

# add a second input channel just to make it less trivial
x[:, 30:70, 1] = 0.5

# ----------------------------
# Run model
# ----------------------------
with torch.no_grad():
    out = model(x, noise=False)
    states = out["states"][0]   # [T, n_rec]

# ----------------------------
# Convert to numpy for plotting
# ----------------------------
W_rec = model.W_rec.detach().cpu().numpy()
W_inp = model.W_inp.detach().cpu().numpy()
states_np = states.detach().cpu().numpy()
x_np = x[0].detach().cpu().numpy()


In [ ]:
# choose 3 time points
time_points = [10, 50, 90]

# ----------------------------
# Plot
# ----------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# W_rec
im0 = axes[0, 0].imshow(W_rec, aspect="auto")
axes[0, 0].set_title("Recurrent weights W_rec")
axes[0, 0].set_xlabel("Presynaptic neuron")
axes[0, 0].set_ylabel("Postsynaptic neuron")
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

# W_inp
im1 = axes[0, 1].imshow(W_inp, aspect="auto")
axes[0, 1].set_title("Input weights W_inp")
axes[0, 1].set_xlabel("Input neuron")
axes[0, 1].set_ylabel("Recurrent neuron")
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

# Input over time
im2 = axes[0, 2].imshow(x_np.T, aspect="auto")
axes[0, 2].set_title("Input x(t)")
axes[0, 2].set_xlabel("Time step")
axes[0, 2].set_ylabel("Input neuron")
plt.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)

# Full neural activity
im3 = axes[1, 0].imshow(states_np.T, aspect="auto")
axes[1, 0].set_title("Hidden activity h(t)")
axes[1, 0].set_xlabel("Time step")
axes[1, 0].set_ylabel("Neuron")
for t in time_points:
    axes[1, 0].axvline(t, linestyle="--")
plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)

# Activity at 3 time points
for t in time_points:
    axes[1, 1].plot(states_np[t], label=f"t={t}")
axes[1, 1].set_title("Neuron activity at selected time points")
axes[1, 1].set_xlabel("Neuron index")
axes[1, 1].set_ylabel("Activity")
axes[1, 1].legend()

# Mean population activity over time
axes[1, 2].plot(states_np.mean(axis=1))
axes[1, 2].set_title("Mean population activity")
axes[1, 2].set_xlabel("Time step")
axes[1, 2].set_ylabel("Mean activity")
for t in time_points:
    axes[1, 2].axvline(t, linestyle="--")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from core import (
    RNNConfig,
    MotorCortexRNN,
    BCIDecoderConfig,
    PopulationBCIDecoder,
)

# ----------------------------
# 1. Build a real network
# ----------------------------
rnn_cfg = RNNConfig(
    n_inp=10,
    n_rec=200,
    dt=0.01,
    tau=0.1,
    noise_std=0.01,
    w_inp_scale=0.1,
    w_rec_scale=0.5,
    device="cpu",
)

dec_cfg = BCIDecoderConfig(
    n_rec=200,
    n_bci=50,
    device="cpu",
)

rnn = MotorCortexRNN(rnn_cfg)
decoder = PopulationBCIDecoder(dec_cfg)

# ----------------------------
# 2. Make simple trial inputs
# ----------------------------
batch_size = 20
T = 120
x = torch.zeros(batch_size, T, rnn_cfg.n_inp)

# baseline epoch
x[:, :30, 0] = 0.2

# task epoch
x[:, 30:90, 0] = 1.0
x[:, 30:90, 1] = 0.5

# late epoch
x[:, 90:, 0] = 0.4
x[:, 90:, 1] = 0.2

# small trial-by-trial noise
x += 0.05 * torch.randn_like(x)

# ----------------------------
# 3. Run network
# ----------------------------
with torch.no_grad():
    out = rnn(x, noise=False)
    states = out["states"]          # [batch, time, 200]
    cursor = decoder(states)        # [batch, time]

# Flatten all hidden states for PCA
states_flat = states.reshape(-1, rnn_cfg.n_rec).cpu().numpy()   # [batch*T, 200]

# ----------------------------
# 4. PCA on hidden activity
# ----------------------------
pca = PCA(n_components=2)
states_pc = pca.fit_transform(states_flat)   # [batch*T, 2]

# Project decoder axis into the PCA plane
axis = decoder.axis.cpu().numpy()            # [200]
axis_pc = pca.components_ @ axis             # [2]

# Also get the mean hidden state for centering the arrow visually
mean_pc = states_pc.mean(axis=0)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Decoder axis in PC space:", axis_pc)

# ----------------------------
# 5. Plot
# ----------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# A. all activity in PCA space
axes[0].scatter(states_pc[:, 0], states_pc[:, 1], s=8, alpha=0.35)
axes[0].quiver(
    mean_pc[0], mean_pc[1],
    axis_pc[0], axis_pc[1],
    angles="xy", scale_units="xy", scale=1, width=0.005
)
axes[0].set_title("All hidden states in PC1-PC2")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].axhline(0, linewidth=0.5)
axes[0].axvline(0, linewidth=0.5)

# B. one example trial trajectory
trial_idx = 0
trial_states = states[trial_idx].cpu().numpy()                 # [T, 200]
trial_pc = pca.transform(trial_states)                         # [T, 2]

axes[1].plot(trial_pc[:, 0], trial_pc[:, 1], marker="o", markersize=2, linewidth=1)
axes[1].quiver(
    mean_pc[0], mean_pc[1],
    axis_pc[0], axis_pc[1],
    angles="xy", scale_units="xy", scale=1, width=0.005
)
axes[1].set_title("One trial trajectory in PC1-PC2")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].axhline(0, linewidth=0.5)
axes[1].axvline(0, linewidth=0.5)

# C. cursor for that trial
trial_cursor = cursor[trial_idx].cpu().numpy()
axes[2].plot(trial_cursor)
axes[2].set_title("Decoder output (cursor) for one trial")
axes[2].set_xlabel("Time step")
axes[2].set_ylabel("Cursor value")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from core import TrialInputConfig, generate_trial_inputs

cfg = TrialInputConfig(n_inp=10)
out = generate_trial_inputs(batch_size=3, cfg=cfg, noise=True)

x = out["x"][0].cpu().numpy()   # one trial

plt.figure(figsize=(10, 4))
plt.imshow(x.T, aspect="auto")
plt.xlabel("Time step")
plt.ylabel("Input channel")
plt.title("One generated trial input")
plt.colorbar()
plt.show()

print(out["epoch_slices"])

In [ ]:
import matplotlib.pyplot as plt

trial = generate_trial_inputs(batch_size=1, cfg=TrialInputConfig())
x = trial["x"]

out = rnn(x, noise=False)
states = out["states"]          # [1, T, 200]
cursor = decoder(states)[0]     # [T]

plt.plot(cursor.detach().cpu().numpy())
plt.xlabel("Time step")
plt.ylabel("Cursor")
plt.title("Cursor trajectory for one trial")
plt.show()

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from core import (
    RNNConfig,
    MotorCortexRNN,
    BCIDecoderConfig,
    PopulationBCIDecoder,
    TrialInputConfig,
    CursorTargetConfig,
    TrainerConfig,
    BCITrainer,
)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

rnn = MotorCortexRNN(
    RNNConfig(
        n_inp=10,
        n_rec=200,
        noise_std=0.01,
        device="cpu",
    )
)

decoder = PopulationBCIDecoder(
    BCIDecoderConfig(
        n_rec=200,
        n_bci=50,
        device="cpu",
    )
)

trial_cfg = TrialInputConfig(
    n_inp=10,
    t_baseline=30,
    t_task=60,
    t_late=30,
    baseline_scale=0.3,
    task_scale=1.0,
    late_scale=0.6,
    input_noise_std=0.01,
    device="cpu",
)

trainer_cfg = TrainerConfig(
    train_mode="input",      # first prove the system can learn
    lr=1e-3,
    grad_clip_norm=1.0,
    device="cpu",
    freeze_trial_inputs=True,
)


target_cfg = CursorTargetConfig(
    baseline_target=0.0,
    late_target=1.0,
    task_target_mode="linear",
    baseline_weight=0.5,
    task_weight=1.0,
    late_weight=2.0,
    device="cpu",
)

trainer = BCITrainer(
    rnn=rnn,
    decoder=decoder,
    trial_cfg=trial_cfg,
    target_cfg=target_cfg,
    trainer_cfg=trainer_cfg,
)

history = trainer.fit(
    n_steps=100,
    batch_size=32,
    eval_every=10,
    eval_batch_size=64,
    print_every=10,
)

'''history = trainer.fit(
    n_steps=400,
    batch_size=32,
    eval_every=20,
    eval_batch_size=64,
    input_noise_train=True,
    rnn_noise_train=False,
    input_noise_eval=False,
    rnn_noise_eval=False,
    print_every=20,
)'''

hist = history.to_dict()

train_steps = [s for s, m in zip(hist["step"], hist["mode"]) if m == "train"]
train_loss = [l for l, m in zip(hist["loss"], hist["mode"]) if m == "train"]
eval_steps = [s for s, m in zip(hist["step"], hist["mode"]) if m == "eval"]
eval_loss = [l for l, m in zip(hist["loss"], hist["mode"]) if m == "eval"]

plt.figure(figsize=(8, 4))
plt.plot(train_steps, train_loss, label="train")
plt.plot(eval_steps, eval_loss, label="eval")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training history")
plt.legend()
plt.show()

eval_out = trainer.evaluate(batch_size=1, input_noise=False, rnn_noise=False)
cursor = eval_out["cursor"][0].detach().cpu().numpy()
target = eval_out["target"][0].detach().cpu().numpy()

plt.figure(figsize=(8, 4))
plt.plot(cursor, label="cursor")
plt.plot(target, label="target")
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Cursor vs target")
plt.legend()
plt.show()